In [ ]:
# Import Dataset
from google.colab import files
uploaded = files.upload()
import sqlite3
import pandas as pd
df = pd.read_csv('pharmacy_dataset.csv')

# Merapikan format tanggal
df['expiry_date'] = pd.to_datetime(df['expiry_date'], format='mixed').dt.strftime('%Y-%m-%d')

# Dataset Memory
conn = sqlite3.connect(':memory:')
df.to_sql('obat', conn, index=False, if_exists='replace')

print(f"Data berhasil diimport: {len(df)} baris")

Saving pharmacy_dataset.csv to pharmacy_dataset.csv
Data berhasil diimport: 2042 baris


In [4]:
# query1: Obat yang akan kedaluwarsa dalam periode tertentu
query1 = '''
SELECT drug_name, category, expiry_date, stock_units
FROM obat
WHERE expiry_date BETWEEN '2026-01-01' AND '2026-06-30'
ORDER BY expiry_date ASC
'''
pd.read_sql(query1, conn)

,drug_name,category,expiry_date,stock_units
0,Tretinoin,Dermatology,2026-01-01,169
1,Desloratadine,Antihistamines,2026-01-02,41
2,Tizanidine,Painkillers,2026-01-03,223
3,Levothyroxine,General,2026-01-03,73
4,Tramadol 50mg,Painkillers,2026-01-04,453
...,...,...,...,...
354,Loratadine,Antihistamines,2026-06-27,260
355,Ibuprofen,General,2026-06-28,209
356,Gliclazide,Antidiabetics,2026-06-29,196
357,Indomethacin,Painkillers,2026-06-29,138


In [5]:
# query2: Obat populer dengan stok kosong/hampir kosong
query2 = '''
SELECT drug_name, category, stock_units, is_popular, is_available
FROM obat
WHERE is_popular = 1 AND is_available = 0
ORDER BY drug_name
'''
pd.read_sql(query2, conn)

,drug_name,category,stock_units,is_popular,is_available
0,Loratadine,Antihistamines,0,1,0
1,Paracetamol 1g,Painkillers,0,1,0


In [6]:
# query3: Obat termurah dan termahal per kategori
query3 = '''
SELECT
    category,
    MIN(price_egp) AS harga_termurah,
    MAX(price_egp) AS harga_termahal
FROM obat
GROUP BY category
ORDER BY harga_termahal DESC
'''
pd.read_sql(query3, conn)

,category,harga_termurah,harga_termahal
0,Antidiabetics,28.03,595.25
1,Cardiovascular,42.76,593.52
2,Women Health,31.95,422.45
3,Respiratory,60.00,350.00
4,Antibiotics,30.28,340.88
5,Painkillers,15.93,324.75
6,Dermatology,25.27,316.56
7,Drops,31.31,299.05
8,Vitamins & Supplements,20.03,279.31
9,Neurology,85.00,275.00


In [7]:
# query4: Mencari semua obat berbentuk "Tablet"
query4 = '''
SELECT drug_name, category, form, price_egp
FROM obat
WHERE form LIKE '%Tablet%'
ORDER BY category
'''
pd.read_sql(query4, conn)

,drug_name,category,form,price_egp
0,Doxycycline,Antibiotics,Tablet,63.50
1,Gentamicin,Antibiotics,Tablet,265.21
2,Amoxicillin,Antibiotics,Tablet,40.54
3,Nitrofurantoin,Antibiotics,Tablet,216.00
4,Cefixime,Antibiotics,Tablet,40.02
...,...,...,...,...
595,Alendronate,Women Health,Vaginal Tablet,52.23
596,Folic_Acid 5mg,Women Health,Tablet,219.53
597,Tibolone,Women Health,Tablet,135.93
598,Prenatal Vitamins,Women Health,Vaginal Tablet,120.75
